<a href="https://colab.research.google.com/github/Moulicodes/movie-recommendation-system/blob/main/movie_recommendation_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df_movies = pd.read_csv('/content/tmdb_5000_movies.csv')

In [ ]:
df_movies.head(1)

In [ ]:
df_movies.head(1)['genres'].values

In [ ]:
df_movies.head(1)['keywords'].values

In [ ]:
df_movies.head(1)['production_companies'].values

In [ ]:
df_movies.head(1)['production_countries'].values

In [ ]:
df_movies['spoken_languages'].value_counts()

In [ ]:
df_movies['status'].value_counts()

In [ ]:
df_movies[df_movies['original_title'] != df_movies['title']].shape

In [ ]:
df_movies[['vote_average', 'vote_count', 'popularity', 'title']].sample(5)

In [ ]:
sns.histplot(df_movies['original_language'])
plt.xticks(rotation = 'vertical')
plt.show()

In [ ]:
import ast

df_movies['keywords'] = df_movies['keywords'].apply(ast.literal_eval)
df_movies['genres'] = df_movies['genres'].apply(ast.literal_eval)
df_movies['production_companies'] = df_movies['production_companies'].apply(ast.literal_eval)


In [ ]:
def transform_listofdicts(listofdicts):
  str = ""
  for dict in listofdicts:
    str += dict['name'].lower().replace(" ", "") + " "
  return str.strip()

In [ ]:
df_movies['genres'] = df_movies['genres'].apply(transform_listofdicts)
df_movies['keywords'] = df_movies['keywords'].apply(transform_listofdicts)
df_movies['production_companies'] = df_movies['production_companies'].apply(transform_listofdicts)

In [ ]:
df_movies = df_movies.drop(columns = ['budget', 'homepage', 'original_language', 'original_title', 'production_countries', 'release_date', 'revenue', 'spoken_languages', 'status', 'vote_count' ])
df_movies.shape

In [ ]:
df_movies.head(1)

In [ ]:
df_credits = pd.read_csv('/content/tmdb_5000_credits.csv')

In [ ]:
df_credits.head()

In [ ]:
df_credits['cast'] = df_credits['cast'].apply(ast.literal_eval).apply(transform_listofdicts)
df_credits['crew'] = df_credits['crew'].apply(ast.literal_eval).apply(transform_listofdicts)

In [ ]:
df_credits.shape

In [ ]:
df_movies.head()

In [ ]:
df_credits.head()

In [ ]:
df = df_movies.merge(df_credits, left_on = 'id', right_on = 'movie_id')
df = df.drop(columns = ['movie_id', 'title_y'])
df = df.rename(columns = {'title_x': 'title'})
df.head(1)

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df[["overview", "tagline"]] = df[["overview", "tagline"]].fillna("")
df["runtime"] = df["runtime"].fillna(0)

In [ ]:
df.info()

In [ ]:
df.head(3)

In [ ]:
!pip install nltk

In [ ]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')


In [ ]:
stopwords.words('english')

In [ ]:
import string
string.punctuation

In [ ]:
def preprocess_text(text):
  text = text.lower()
  words = nltk.word_tokenize(text)
  processed_text = ""
  for word in words:
    if (word not in stopwords.words('english')) and (not word.isnumeric()) and (word not in string.punctuation):
      processed_text += word + " "

  return processed_text.strip()

In [ ]:
# preprocess_text('In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.')

df['overiew'] = df['overview'].apply(preprocess_text)
df['overiew'].head()

In [ ]:
df['tagline'] = df['tagline'].apply(preprocess_text)
df['tagline'].head()

In [ ]:
df['processed_description'] = df['genres'] + " " + df['keywords'] + " " + df['overview'] + " " + df['tagline'] + " " + df['cast'] + df['crew']
df['processed_description'].head(1).values

1. TF-IDF Vectorizer

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words = 'english')
tfidf_matrix = tfidf.fit_transform(df['processed_description'])
tfidf_matrix.shape

Because we are using tfidf vectorizer there's no need to explicitly remove stop words, punctuations or convert texts to lower case as it will handle all these functions by itself. But we still need to explicitly implement space collapsing wherever necessary. For example, removing the space between first and last name.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

tfidf_cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
tfidf_cosine_sim_rounded = tfidf_cosine_sim.round(3)
tfidf_cosine_sim_rounded.shape

In [ ]:
df.iloc[[2]][['title', 'genres', 'keywords', 'overview', 'tagline', 'production_companies', 'cast', 'crew']]

In [ ]:
np.where((tfidf_cosine_sim_rounded[2] > 0.1) & (tfidf_cosine_sim_rounded[2] < 0.99))[0]

In [ ]:
tfidf_cosine_sim_rounded[2][(tfidf_cosine_sim_rounded[2] > 0.1) & (tfidf_cosine_sim_rounded[2] < 0.99)]

In [ ]:
df.iloc[[11, 29]][['title', 'genres', 'keywords', 'overview', 'tagline', 'production_companies', 'cast', 'crew']]

**Problem with tfidf vectorizer:**

It penalizes the words present in genres, keywords, cast and crew if they occur very frequently in multiple movies. But the more matching the genres, keywords, cast and crew of various movies, the greater should be the similarity scores between them. But because tfidf penalizes the frequently occuring words, it behaves counter-productive to our goal.**bold text**

2. Count Vectorizer

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer(stop_words = 'english')
count_matrix = count_vectorizer.fit_transform(df['processed_description'])
count_matrix.shape

In [ ]:
count_cosine_sim = cosine_similarity(count_matrix, count_matrix)
count_cosine_sim_rounded = count_cosine_sim.round(3)
count_cosine_sim_rounded.shape

In [ ]:
df.iloc[[2]][['title', 'genres', 'keywords', 'overview', 'tagline', 'production_companies', 'cast', 'crew']]

In [ ]:
np.where((count_cosine_sim_rounded[2] > 0.1) & (count_cosine_sim_rounded[2] < 0.99))[0]

In [ ]:
count_cosine_sim_rounded[2][(count_cosine_sim_rounded[2] > 0.1) & (count_cosine_sim_rounded[2] < 0.99)]

So, there are four other movies that share a decent similarity score with the movie Spectre.

In [ ]:
df.iloc[np.where((count_cosine_sim_rounded[2] > 0.1) & (count_cosine_sim_rounded[2] < 0.99))[0]][['title', 'genres', 'keywords', 'overview', 'tagline', 'production_companies', 'cast', 'crew']]

**Problem with count vectorizer:**

Consider two completely unrelated action movies.

Movie A (James Bond): "A man goes on a mission to save the world from danger."

Movie B (Avatar): "A man on a foreign planet tries to save his new world from danger."

Both movies heavily feature highly common narrative words: "man", "save", "world", "danger"

So, both of these movies are likely to have high similarity scores even though they are completely unrelated films

A better solution:
*   Use count vectorizer for genres, keywords, cast, crew and columns where the greater the occurence of similar words in various movies higher is the similarity between them.
*   Use tf-idf vectorizer for overview, tagline and columns where the context of the entire text and the excluisivity of certain words decide the similarity scores.
*   Before applying tf-idf vectorizer, apply stemming to the columns we have considered for tf_idf vectorization.




In [158]:
df['cv_col'] = df['genres'] + " " + df['keywords'] + " " + df['cast'] + " " + df['crew']
df['tfidf_col'] = df['overview'] + " " + df['tagline']

In [159]:
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()
df['tfidf_col'] = df['tfidf_col'].apply(lambda x: " ".join([ps.stem(word) for word in x.split()]))

In [160]:
df['tfidf_col'].head()

,tfidf_col
0,"in the 22nd century, a parapleg marin is dispa..."
1,"captain barbossa, long believ to be dead, ha c..."
2,a cryptic messag from bond’ past send him on a...
3,follow the death of district attorney harvey d...
4,"john carter is a war-weary, former militari ca..."


In [161]:
count_vec = CountVectorizer(stop_words = 'english')
count_mat = count_vec.fit_transform(df['cv_col'])

tfidf_vec = TfidfVectorizer(stop_words = 'english')
tfidf_mat = tfidf_vec.fit_transform(df['tfidf_col'])

In [163]:
from scipy.sparse import hstack

combined_mat = hstack([count_mat, tfidf_mat])
hybrid_sim = cosine_similarity(combined_mat, combined_mat).round(3)

In [231]:
def recommend_movies(movie_index, top_n=5):
  print("Movie:", df.iloc[movie_index]['title'])


  similarity_scores = hybrid_sim[movie_index]

  # [::-1] reverses the array so the highest scores come first
  # We dont start from 0 (i.e, [1:top_n+1]) because the highest score (1.0) is the movie itself!
  sorted_indices = np.argsort(similarity_scores)[::-1][1:top_n+1]

  print(f"Top Similarity Scores: {similarity_scores[sorted_indices]}")
  print(f"Top {top_n} Recommended Movies:")
  return df.iloc[sorted_indices][['title', 'genres', 'keywords', 'cast', 'crew']]

In [232]:
recommend_movies(0)

Movie: Avatar
Top Similarity Scores: [0.139 0.124 0.104 0.088 0.071]
Top 5 Recommended Movies:


,title,genres,keywords,cast,crew
282,True Lies,action thriller,spy terrorist florida gun kidnapping horseback...,arnoldschwarzenegger jamieleecurtis tomarnold ...,markgoldblatt malifinn cindycarr jamescameron ...
25,Titanic,drama romance thriller,shipwreck iceberg ship panic titanic oceanline...,katewinslet leonardodicaprio francesfisher bil...,malifinn jameshorner jamescameron jamescameron...
2403,Aliens,horror action thriller sciencefiction,android extraterrestrialtechnology spacemarine...,sigourneyweaver michaelbiehn jamesremar paulre...,michaellamont raylovejoy robinclarke janefeinb...
279,Terminator 2: Judgment Day,action thriller sciencefiction,cyborg shotgun post-apocalyptic dystopia moral...,arnoldschwarzenegger lindahamilton robertpatri...,dodydorn johnm.dwyer galeannehurd markgoldblat...
19,The Hobbit: The Battle of the Five Armies,action adventure fantasy,corruption elves dwarves orcs middle-earth(tol...,martinfreeman ianmckellen richardarmitage kens...,howardshore christopherboyes peterjackson pete...


In [233]:
recommend_movies(1)

Movie: Pirates of the Caribbean: At World's End
Top Similarity Scores: [0.627 0.274 0.27  0.182 0.135]
Top 5 Recommended Movies:


,title,genres,keywords,cast,crew
12,Pirates of the Caribbean: Dead Man's Chest,adventure fantasy action,witch fortuneteller bondage exoticisland monst...,johnnydepp orlandobloom keiraknightley stellan...,dariuszwolski goreverbinski jerrybruckheimer t...
17,Pirates of the Caribbean: On Stranger Tides,adventure action fantasy,sea captain mutiny sword primeminister sailing...,johnnydepp penélopecruz ianmcshane kevinmcnall...,dariuszwolski jerrybruckheimer tedelliott terr...
199,Pirates of the Caribbean: The Curse of the Bla...,adventure fantasy action,exoticisland blacksmith eastindiatradingcompan...,johnnydepp geoffreyrush orlandobloom keiraknig...,arthurschmidt dariuszwolski klausbadelt goreve...
13,The Lone Ranger,action adventure western,texas horse survivor texasranger partner outla...,johnnydepp armiehammer williamfichtner helenab...,goreverbinski goreverbinski jerrybruckheimer t...
178,Rango,animation comedy family western adventure,sheriff nevada pet rango chameleon lasvegas ca...,johnnydepp islafisher nedbeatty billnighy alfr...,goreverbinski goreverbinski goreverbinski hans...


In [234]:
recommend_movies(2)

Movie: Spectre
Top Similarity Scores: [0.26  0.172 0.131 0.1   0.098]
Top 5 Recommended Movies:


,title,genres,keywords,cast,crew
29,Skyfall,action adventure thriller,spy secretagent sociopath killer artgallery br...,danielcraig judidench javierbardem ralphfienne...,thomasnewman sammendes annapinnock rogerdeakin...
11,Quantum of Solace,adventure action thriller crime,killing undercover secretagent britishsecretse...,danielcraig olgakurylenko mathieuamalric judid...,paulhaggis dennisgassner ianfleming louisefrog...
147,Die Another Day,adventure action thriller,laser britishsecretservice secretserviceagent ...,piercebrosnan halleberry rosamundpike rickyune...,ianfleming leetamahori barbarabroccoli robertw...
170,The World Is Not Enough,adventure action thriller,british mission oil heiress bilbaospain britis...,piercebrosnan sophiemarceau robertcarlyle deni...,ianfleming peterlamont davidarnold adrianbiddl...
277,Casino Royale,adventure action thriller,italy poker casino terrorist banker money free...,danielcraig evagreen madsmikkelsen judidench j...,michaellamont paulhaggis stuartbaird ianflemin...


In [235]:
recommend_movies(4535)

Movie: Seven Samurai
Top Similarity Scores: [0.152 0.086 0.083 0.053 0.051]
Top 5 Recommended Movies:


,title,genres,keywords,cast,crew
2908,Madadayo,drama,,tatsuomatsumura kyôkokagawa hisashiigawa georg...,akirakurosawa akirakurosawa takaosaitô shôjiue...
1877,Tora! Tora! Tora!,history action drama adventure war,japan worldwarii pearlharbor soldier imperialj...,martinbalsam sôyamamura josephcotten tatsuyami...,pembrokej.herring jerrygoldsmith richardfleisc...
4638,Amidst the Devil's Wings,drama action crime,,,
2646,Silent Trigger,drama action thriller,,dolphlundgren ginabellman conraddunn christoph...,russellmulcahy sergioaltieri
3945,Love's Abiding Joy,tvmovie action drama family,,erincottrell loganbartholomew williammorganshe...,michaellandonjr.
